In [2]:
import gmpy2 as gmp
import random
import libs.Parameters as par
from libs.Tools import HashToField,Fpsgn0

def rsqrt(x,p):
      if p & 3==3:
           _=pow(x,(p+1)>>2,p) 
           __=((_**2)*pow(x,-1,p)) % p
           if __!=1: return None ## Check if 'x' is a quadratic Residu modulo p using Euler's Criterion
           else: return _
      else:   ## If the Modulo is in the form 4k+1, we use the Tonelli-Shanks Algorithm         
        if pow(x,(p-1)>>1,p)!=1:return None
        else:         
          g=gmp.xmpz(2)
          while (pow(g,(p-1)>>1,p)!=p-1):g+=1
          e1,e2 =(p-1)>>1,gmp.xmpz(0)
          while (e1 & 1==0):
              e1>>=1
              e2>>=1
              if (pow(x,e1,p)*pow(g,e2,p))%p==p-1:e2=e2+((p-1)>>1)
          return pow(x,(e1+1)>>1,p)*pow(g,e2>>1,p)
        
def ECFP(CurveParams):
   
   # Curves are implemented using Affine coordinates for the table and projective coordinates during point's multiplication
   # Mixed addition is used when adding to the lookup-table Elements
   # Affine coordinates represtentation is technically faster than Jacobian under Python, since modular inversion using GMP has a cost near to multiplication by a field's element.
   _addProjective = lambda _x,_y,_z,xx,yy:[xx,yy,1]if _z==0 else [((_2:=xx*_z-_x)*(_6:=((_1:=yy*_z-_y)**2)*_z-(_4:=_2*(_3:=_2**2))-2*(_5:=_3*_x)))% _basefield,(_1*(_5-_6)-_4*_y)% _basefield,(_4*_z)% _basefield]
   _addAffine     = lambda _x,_y,xx,yy:[(_1:=((_0:=((_y-yy)*gmp.invert((_x-xx),_basefield)))**2-_x-xx) % _basefield),(_0*(_x-_1)-_y) % _basefield]
   _doubleProjective = lambda _x,_y,_z:[_x,_y,_z]if _z==0 else [((_6:=(_1:=3*(_0:=_x**2))**2-((_5:=(_x+(_3:=_y*(_2:=(_y*_z<<1))))**2-_0-(_4:=_3**2))<<1))*_2) % _basefield,(_1*(_5-_6)-2*_4)% _basefield,(_2*_2**2)% _basefield]
   _doubleAffine     = lambda _x,_y:[(_1:=((_0:=((3*_x**2)*gmp.invert((_y<<1),_basefield)))**2-2*_x) % _basefield),(_0*(_x-_1)-_y)% _basefield]

   def _recordOneScalar(x):
      mu =  x.bit_length()      
      mu = mu + _wSize - (mu % _wSize)
      x =  x | (1 << mu)
      Code =  1
      while (x != 1):
            sign  = ((x >> (_wSize - 1)) & 2) - 1    
            Code  = (Code << _wSize) | ( (((x ^ sign) & _wMask) | 1) + (sign >> 1))
            x    =  x        >> _wSize 
      return Code >> 1

   def _swumapping(u):
      #     Simplified Shallue-van de Woestijne-Ulas Method (Simplified SWU for AB == 0)
      #     https://datatracker.ietf.org/doc/html/draft-irtf-cfrg-hash-to-curve-05#section-6.6.3
      #     https://datatracker.ietf.org/doc/html/draft-irtf-cfrg-hash-to-curve-05#appendix-C.2
      t1 = (_swuZ * u**2) % _basefield
      t2 = t1**2 % _basefield
      x1 = (gmp.invert(t1 + t2,_basefield))
      if x1==0 : x1 = _invZ
      else :x1 = (x1 + 1) % _basefield
      x1  = (x1 * _BdivA) % _basefield
      gx1 = ((x1**2 + _swuA) * x1 + _swuB) % _basefield
      y   = rsqrt(gx1, _basefield)
      if (y != None):
         if Fpsgn0(u, _basefield) != Fpsgn0(y, _basefield): y = -y % _basefield
         x = x1
      else:
         x2  = (t1 * x1) % _basefield
         t2  = (t1 * t2) % _basefield
         gx2 = (gx1 * t2) % _basefield
         y   = rsqrt(gx2, _basefield)
         if Fpsgn0(u, _basefield) != Fpsgn0(y, _basefield): y = -y % _basefield
         x=x2
      #     11_isogeny map from E' to E "Wahby and Boneh" https://eprint.iacr.org/2019/403
      x_num, x_den, y_num, y_den = _Xnum[0], _Xden[0], _Ynum[0], _Yden[0]
      powx = x
      for i in range(1,len(_Ynum)):
         y_num = (y_num + _Ynum[i]*powx) % _basefield
         y_den = (y_den + _Yden[i]*powx) % _basefield
         if i<len(_Xnum):
            x_num = (x_num + (_Xnum[i] * powx)) % _basefield
            x_den = (x_den + (_Xden[i] * powx)) % _basefield
         powx = powx * x
      return  [(x_num * gmp.invert(x_den,_basefield)) % _basefield ,(y*(y_num * gmp.invert(y_den,_basefield))) % _basefield]


   # Here begans the part covered by the works in tha paper :"Optimizing and securing GLV multiplication over BLS pairings-friendly curves"
   # Three Lookup-table construction Algorithms , with correspoding recording/aligning schemes : 
   #                       -  Standard versnion, 
   #                       -  First Proposed approach (computation half the table using Endomorphism);
   #                       - Space-optimized lookup table (Only half the lookup table is generated)  

   def _recordScalar_Algo4(scalar):        
      x1 , x2 = scalar % _Lamda , scalar // _Lamda
      _beta = (~x1 & 1) * (_Lamda & 1)
      x1 = x1 + _beta * _Lamda
      x2 = x2 - _beta      
      mu = (x1 | abs(x2)).bit_length()
      mu = mu + _wDSize - (mu % _wSize)
      x1 = x1 | (1 << mu)   
      code =  1
      while (x1 != 1):
            sign = ((x1 >> _wSize) & 1) - 1
            ai   = ((x1 ^ sign) |   1 ) & _wMask 
            bi   = ((x2 + sign) ^ sign) & _wMask
            code = (code << _wDSize) | ((bi << _wSize) + ai  + sign)
            sign = sign ^ ((~(bi + _wMask) >> _wSize) & sign)   
            x1   =  x1  >> _wSize 
            x2   = (x2  >> _wSize) + sign 
      return code >> 3 
   
   def _recordScalar_Algo9(scalar):
      x1 , x2 = scalar % _Lamda , scalar // _Lamda
      _beta = (~x1 & 1) * (_Lamda & 1)
      x1 = x1 + _beta * _Lamda
      x2 = x2 - _beta                       
      mu = (x1 | abs(x2)).bit_length()
      mu = mu + _wDSize - (mu % _wSize)
      x1 = x1 | (1 << mu)   
      code =  1
      while (x1 != 1):
         sign = ((x1 >> _wSize) & 1) - 1                 
         even = ~x2 & 1                                  
         ai   = ((x1 ^ sign) |  1  ) & _wMask 
         bi   = ((x2 + sign) ^ sign) & _wMask
         di   =  ai - bi
         inc  = ((_wMask - (sign | 1) * di) & (bi + _wMask)) >> _wSize
         inc  =  inc * even + ((even - 1) * sign)  
         code = (code << _wDSize) | ((bi << _wSize) + ai  + sign)
         x1   = (x1 >> _wSize) 
         x2   = (x2 >> _wSize) + inc
      return code >> 3
   
   def _recordScalar_Algo12(scalar):
      x1 , x2 = scalar % _Lamda , scalar // _Lamda
      _beta = (~x1 & 1) * (_Lamda & 1)
      x1 = x1 + _beta * _Lamda
      x2 = x2 - _beta      
      mu = (x1 | abs(x2)).bit_length()
      mu = mu + _wDSize - (mu % _wSize)
      x1 = x1 | (1 << mu)   
      code =  1
      while (x1 != 1):
         sign = ((x1 >> _wSize) & 1) - 1                 
         even = ~x2 & 1                                  
         ai   = ((x1 ^ sign) |  1  ) & _wMask 
         bi   = ((x2 + sign) ^ sign) & _wMask
         di   =  ai - bi
         inc  = ((_wMask - (sign | 1) * di) & (bi + _wMask)) >> _wSize
         ai   = (ai - bi * even) & _wMask
         bi   =  di * even + bi
         inc  =  inc * even + ((even - 1) * sign)  
         code = (code << _wDSize) | (even + ((sign + 1) << 1) + (((ai >> 1) + ((bi - 1) << 1)) << 2))  
         x1   = (x1 >> _wSize) 
         x2   = (x2 >> _wSize) + inc
      return code
   
   #  First Approach :
   #  GLV Multiplication using the proposed unified Recording/Alignment scheme and Standard lookup Table Construction

   def _GLV2PointMulG1_App1(a,P):
      
      # Generating standard version of the lookup table : 
      # Algorithm 5 from the paper: "Optimizing and securing GLV multiplication over BLS pairings-friendly curves".
      _2P  = _doubleAffine(P.x, P.y)
      _phiP = [(P.x * _Wx) % _basefield, P.y]
      T = [[P.x, P.y]]
      T = T + [_addAffine(_2P[0], _2P[1], P.x, P.y)]
      T = T + [_addAffine(_2P[0], _2P[1], T[1][0], T[1][1])]
      T = T + [_addAffine(_2P[0], _2P[1], T[2][0], T[2][1])]
      for _ in range(_tableSize - _blockSize): 
            T = T + [_addAffine(_phiP[0], _phiP[1], T[-_blockSize][0],T[-_blockSize][1] )]

      # Regular scalar decomposition using proposed approach from section 3.4: Algorithm 14
      _scalar = a % _r
      _alpha  = (_Lamda | _scalar) & 1
      a     = (_r - _scalar) * (1 - _alpha) + _scalar * _alpha   
      _code = _recordScalar_Algo4(a)

      # Proposed regular multiplication Loop for the Second Approach :Algorithm 6
      _aP   = T[(_code & _wMask) << 2] + [1]     
      _code = _code >> _wSize
      while (_code != 1):
               _sig  = 2 * (_code & 1) - 1
               _idx  = (_code & _wDMask) >> 1
               _code =  _code >> _wDSize
               _aP = _doubleProjective(_aP[0], _aP[1], _aP[2])
               _aP = _doubleProjective(_aP[0], _aP[1], _aP[2])
               _aP = _doubleProjective(_aP[0], _aP[1], _aP[2])
               _aP = _addProjective(_aP[0], _aP [1], _aP[2], T[_idx][0], _sig * T[_idx][1])   
      _ = gmp.invert(_aP[2], _basefield)
      return ECFP(_aP[0]*_, ((_alpha << 1) - 1) * _aP[1]*_) 

   #  Second Approach :
   #  GLV Multiplication using the proposed lookup table structure :Half of the table elements are computed using Endomorphism
   #  Corresponding recording/alignements scheme is performed using Algorithm 9
   
   def _GLV2PointMulG1_App2(a,P):
      
      # Generating the proposed variant of the lookup table : 
      # Algorithm 8 from the paper: "Optimizing and securing GLV multiplication over BLS pairings-friendly curves".
      T = 32*[[]]
      _2P  = _doubleAffine(P.x, P.y)
      _2phiP = [(_2P[0] * _Wx) % _basefield, _2P[1]]
      T [4]= [(P.x * -(_Wx+1)) % _basefield, -P.y % _basefield] # using Gamma(P)
      T [5]= _addAffine(_2P[0], _2P[1], T[4][0], T[4][1])
      T [6]= _addAffine(_2P[0], _2P[1], T[5][0], T[5][1])
      T [7]= _addAffine(_2P[0], _2P[1], T[6][0], T[6][1])      
      for i in range(_tableSize >> 1):
         id1 = (i & 3) + ((i << 1) & (-8)) + _blockSize
         id2 = ((id1 >> 3) + (((id1 >> 2) - ((id1 & 3) << 1) - 1) << 2)) % _tableSize 
         if i > _blockSize - 1:
            T[id1] = _addAffine(_2phiP[0], _2phiP[1], T[id1 - 8][0],T[id1 - 8][1])
         T[id2] = [(T[id1][0] * _Wx) % _basefield, (-T[id1][1]) ]  # using Phi(P)

      # Regular scalar decomposition using proposed approach from section 3.4: Algorithm 14
      _scalar = a % _r
      _alpha = (_Lamda | _scalar) & 1
      a = (_r - _scalar) * (1 - _alpha) + _scalar * _alpha   

      # Proposed regular multiplication Loop for the Second Approach :Algorithm 6
      _code = _recordScalar_Algo9(a)
      _aP   = T[(_code & _wMask) << 2] + [1]     
      _code = _code >> _wSize
      while (_code != 1):
               _sig  = 2 * (_code & 1) - 1
               _idx  = (_code & _wDMask) >> 1
               _code =  _code >> _wDSize
               _aP = _doubleProjective(_aP[0], _aP[1], _aP[2])
               _aP = _doubleProjective(_aP[0], _aP[1], _aP[2])
               _aP = _doubleProjective(_aP[0], _aP[1], _aP[2])
               _aP = _addProjective(_aP[0], _aP [1], _aP[2], T[_idx][0], _sig * T[_idx][1])   
      _ = gmp.invert(_aP[2], _basefield)
      return ECFP(_aP[0] * _, ((_alpha << 1) - 1) * _aP[1] * _)
  

   #  Space-optimized Approach :
   #  GLV Multiplication using the space-optimized lookup table structure :Half of the table elements only are computed
   #  Corresponding recording/alignements scheme is performed using Algorithm 12
   def _GLV2PointMulG1_App3(a,P):
      
      # Generating the space-optimized variant of the lookup table : 
      # Algorithm 10 from the paper: "Optimizing and securing GLV multiplication over BLS pairings-friendly curves".
      _2P  = _doubleAffine(P.x, P.y)
      _2phiP = [(_2P[0] * _Wx) % _basefield, _2P[1]]
      T = [[(P.x * -(_Wx+1)) % _basefield, (-P.y) % _basefield]]
      T = T + [_addAffine(_2P[0], _2P[1], T[0][0], T[0][1])]
      T = T + [_addAffine(_2P[0], _2P[1], T[1][0], T[1][1])]
      T = T + [_addAffine(_2P[0], _2P[1], T[2][0], T[2][1])]    
      for i in range((_tableSize >> 1) - _blockSize):
         T = T + [_addAffine(_2phiP[0], _2phiP[1], T[-_blockSize][0],T[-_blockSize][1])]
      
      # Regular scalar decomposition using proposed approach from section 3.4: Algorithm 14
      _scalar = a % _r
      _alpha = (_Lamda | _scalar) & 1
      a = (_r - _scalar) * (1 - _alpha) + _scalar * _alpha   

      _code = _recordScalar_Algo12(a)

      # Proposed regular multiplication Loop for the space-optimized Approach :Algorithm 13
      _fi   = _code & 1
      _sig  = (_code & 2) - 1 
      _idx  = (_code & _wDMask) >> 2
      _aP   = [(_Wx * _fi + (1 - _fi)) * T[_idx][0], _sig * (1 - 2 * _fi) * T[_idx][1]]+ [1]     
      _code = _code >> _wDSize
      while (_code != 1):
               _fi    = _code & 1   
               _sig  = (_code & 2) - 1
               _idx  = (_code & _wDMask) >> 2
               _code =  _code >> _wDSize
               _aP = _doubleProjective(_aP[0], _aP[1], _aP[2])
               _aP = _doubleProjective(_aP[0], _aP[1], _aP[2])
               _aP = _doubleProjective(_aP[0], _aP[1], _aP[2])
               _aP = _addProjective(_aP[0], _aP [1], _aP[2],(_Wx * _fi + (1 - _fi)) * T[_idx][0], _sig * (1 - 2 * _fi) * T[_idx][1])   
      _ = gmp.invert(_aP[2], _basefield)
      return ECFP(_aP[0] * _, ((_alpha << 1) - 1) * _aP[1] * _)


   class ECFP :
      def __init__(self,x,y=None,dotest=False):
         if x is None:
            self.infinity = True
            self.x, self.y, self.z = 0, 1, 0
         else:
            if y is None:
              self.x = x % _basefield if type(x)==gmp.mpz else gmp.mpz(x) % _basefield
              _ = rsqrt((x**3+ECFP.B) % _basefield,_basefield)
              if (_ != None) : self.infinity, self.y, self.z = False, _, gmp.mpz(1)
              else :raise TypeError("Invalide curve point parametres ...")
            else:
              self.x = x % _basefield if type(x)==gmp.mpz else gmp.mpz(x) % _basefield
              self.y = y % _basefield if type(y)==gmp.mpz else gmp.mpz(y) % _basefield
              self.z = gmp.mpz(1)
              self.infinity = False
              if (dotest) and (y**2 % _basefield != (x**3+ECFP.B) % _basefield) : raise TypeError("Invalide curve point parametres ...")
      
      def __str__(self):return "("+str(self.x)+" , "+str(self.y)+")" if not self.infinity else "Infinity"
      __repr__ = __str__
      
      def __add__(p,q):
        if p.infinity:
           if q.infinity : return ECFP(None)
           else : return ECFP(q.x,q.y)
        else:
           if q.infinity : return p.copy()
           else:
            if p.x == q.x:
               if p.y == q.y:return ECFP((res := _doubleAffine(p.x, p.y))[0], res[1])
               else : return ECFP(None)
            else : return ECFP((res := _addAffine(p.x,p.y,q.x,q.y))[0], res[1])

      def __neg__(p)  : return ECFP(p.x,-p.y)
      def __sub__(p,q): return p + (-q)
      def __eq__(p,q) : return (p.x == q.x) & (p.y == q.y) if not(p.infinity | q.infinity) else (p.infinity == q.infinity)
      def __ne__(p,q) : return (p.infinity != q.infinity) | (p.x != q.x) | (p.y != q.y)
      def __rmul__(p,b):
         if (type(b) != int) & (type(b) != gmp.mpz) : 
            raise TypeError("Invalide scalar value for multiplication ...")
         else :
            _outSig = abs(b+1) - abs(b)
            b = abs(b)
            if p.infinity: return ECFP(None)
            if b==0 :      return ECFP(None)
            if abs(b)==2 : return ECFP((res:=_doubleAffine(p.x,p.y))[0], _outSig * res[1])
            else:
               T = [_doubleAffine(p.x, p.y)] + [[p.x, p.y]]
               T = T + [_addAffine(T[0][0], T[0][1], p.x, p.y)]
               T = T + [_addAffine(T[0][0], T[0][1], T[2][0], T[2][1])]
               T = T + [_addAffine(T[0][0], T[0][1], T[3][0], T[3][1])]
               _code = _recordOneScalar(b+(b & 1)+1)               
               _aP   = T[(_code & (_wMask >> 1)) + 1]+[1]                
               _code = _code >> (_wSize-1)
               while (_code != 1):
                     _sig = 2 * (_code & 1)-1
                     _idx = ((_code & _wMask) >> 1) + 1
                     _aP  = _doubleProjective(_aP[0], _aP[1], _aP[2])
                     _aP  = _doubleProjective(_aP[0], _aP[1], _aP[2])
                     _aP  = _doubleProjective(_aP[0], _aP[1], _aP[2])
                     _aP  = _addProjective(_aP[0], _aP[1], _aP[2], T[_idx][0], _sig*T[_idx][1])
                     _code =  _code >> _wSize
               _aP  = _addProjective(_aP[0], _aP[1], _aP[2], T[1 - (b & 1)][0], -T[1 - (b & 1)][1])
               if _aP[2] == 0 : return ECFP(None) 
               else:                   
                  _ = gmp.invert(_aP[2], _basefield)
                  return ECFP(_aP[0] * _, _outSig  * _ * _aP[1])      
               
      def phi(self) : return ECFP(self.x*_Wx % _basefield, self.y)
      def gamma(self) : return ECFP((self.x*(-(_Wx+1))) % _basefield, (-self.y) % _basefield)

      def glvMulG1(self,a): 
         match  self.GLVToUse:
            case 1:return _GLV2PointMulG1_App1(a, self)
            case 2:return _GLV2PointMulG1_App2(a, self)
            case 3:return _GLV2PointMulG1_App3(a, self)
      
      def glvMulG1_App1(self,a): return _GLV2PointMulG1_App1(a, self)
      def glvMulG1_App2(self,a): return _GLV2PointMulG1_App2(a, self)
      def glvMulG1_App3(self,a): return _GLV2PointMulG1_App3(a, self)
         
      def copy(p) : return ECFP(p.x,p.y)
      def pickRandomPoint():
          _ = _swumapping(gmp.mpz(random.randint(0, _basefield-1)))
          return ECFP(_[0], _[1])
      def pickTorsionPoint() : return _h * ECFP.pickRandomPoint() # Cofactor cleaning
      def pickRandomScalar() : return gmp.mpz(random.randint(0, ECFP.r-1) | (1 << (ECFP.r.bit_length()-1)))
      def hashtoG1(identifier, mode = 0):
         #     mode=0: Encode to Curve (NU-encode-to-curve), mode=1: Random Oracle model (RO-hash-to-curve)
         #     https://datatracker.ietf.org/doc/html/draft-irtf-cfrg-hash-to-curve-06#name-roadmap
         if mode == 0 :
            _ = _swumapping(gmp.mpz(HashToField(identifier, _basefield, extdegree = 1, count = 1)[0][0]))
            return _h * ECFP(_[0], _[1])
         else :
            _  = HashToField(identifier, _basefield, extdegree = 1, count = 2)
            _1 = _swumapping(gmp.mpz(_[0][0]))
            _2 = _swumapping(gmp.mpz(_[1][0]))
            _1, _2 = _addAffine(_1[0], _1[1], _2[0], _2[1])
            return _h * ECFP(_1,_2)
      def isOnCurve(p) : return (p.y**2 % _basefield == (p.x**3 + ECFP.B) % _basefield)
      def isTorsion(p):
         #     Check if a point P is a Torsion Point
         #     According to https://eprint.iacr.org/2022/352.pdf we have to check that  ψ(P)+lamda*P="Infinity"
         return _Lamda * p == p.phi()

   _basefield = CurveParams["p"]
   _r = CurveParams["r"]
   _u = CurveParams["u"]
   _h = CurveParams["h1bis"]     #  reduced Cofactor (1-u), "Wahby and Boneh" https://eprint.iacr.org/2019/403  , section 5   

   #   Parametres of the Endomorphisme for GLV multiplication
   _Lamda = CurveParams["lamda"]
   _Wx = CurveParams["w"]   

   # Windows signed representation parameter
   _wSize  =  3
   _wDSize =  _wSize << 1
   _wMask  = (1 << _wSize) - 1
   _wDMask = (1 << _wDSize) - 1
   _tableSize = (1 << (_wDSize - 1))
   _blockSize = (1 << (_wSize - 1))
   
   #    Parametres of the constant-time Hash to G1 (swu + Isogeny)
   _swuZ  = CurveParams["swuParamsG1"]["Z"]
   _swuA  = CurveParams["swuParamsG1"]["swuA"]
   _swuB  = CurveParams["swuParamsG1"]["swuB"]
   _BdivA = (-(_swuB* gmp.invert(_swuA,_basefield))) % _basefield
   _invZ  = (-(gmp.invert(_swuZ,_basefield))) % _basefield
   _Xnum  = CurveParams["swuParamsG1"]["Xnum"]
   _Xden  = CurveParams["swuParamsG1"]["Xden"]
   _Ynum  = CurveParams["swuParamsG1"]["Ynum"]
   _Yden  = CurveParams["swuParamsG1"]["Yden"]
   ECFP.u = _u
   ECFP.r = _r
   ECFP.h = _h
   ECFP.B = CurveParams["B"]
   ECFP.Description = CurveParams["Description"]
   ECFP.GLVToUse = 3
   return ECFP

In [3]:
line="---------------------------------------------------------------------------------------------------------------------------------------------"
for i in (par.bls12_381_params, par.bls12_461_params,par.bls24_479_params,par.bls24_559_params,par.bls48_575_params):
    print(line)
    print(i["Description"].split('\n', 1)[0])
    print(line)
    EcFp=ECFP(i)
    P      = EcFp.pickTorsionPoint()
    Scalar = EcFp.pickRandomScalar()
    print("P: ",P)
    print("Scalar :",Scalar)
    Q  = Scalar * P    # Naïve  multiplication without using GLV
    print("[Scalar]P :",Q)
    print("")
    Q1 = P.glvMulG1_App1(Scalar)
    Q2 = P.glvMulG1_App2(Scalar)
    Q3 = P.glvMulG1_App3(Scalar)    
    print("Result of the first Approach (Unified record/Align with Standard lookup Table) :")
    print("Q1 =GLV_Ap1(Scalar,P) :",Q1)
    print("Q = Q1: ",Q==Q1) 
    print("")
    print("Result of the second Approach (Modified lookup Table using Endomorphism mapping) :")
    print("Q2 =GLV_Ap2(Scalar,P) :",Q2)
    print("Q = Q2: ",Q==Q2) 
    print("")
    print("Result of the third Approach (Space-optimized lookup Table) :")
    print("Q3 =GLV_Ap3(Scalar,P) :",Q3)
    print("Q = Q3: ",Q==Q3) 



---------------------------------------------------------------------------------------------------------------------------------------------
BLS12-381 Curve 
---------------------------------------------------------------------------------------------------------------------------------------------
P:  (1216840157639703764795058229703628623097145207651732595896054396756282423703257671162972639515009000306153729876997 , 1317859829545830057413294202447019406459740908872222007822756008973406795489151064600706901271837815484630476614230)
Scalar : 47693511600593531401871452856207888728306170760887005301473898854052435083489
[Scalar]P : (1261230900533774851804648071467769965521014555119052234415907684873187223871055204363298461943387661206052362398930 , 1859795596319112210164864439055169783231970230695707152075366319569745072935418861796961493767707026317017367838033)

Result of the first Approach (Unified record/Align with Standard lookup Table) :
Q1 =GLV_Ap1(Scalar,P) : (12612309005337748